[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChiShengChen/neural-signals-101/blob/main/notebooks/01_what_are_neural_signals.ipynb)

> **Running on Google Colab?** Run the next cell first — it installs everything and
> fetches the helper package. **Running locally (after `make setup`)?** The next
> cell does nothing; just run it and continue.

In [ ]:
# --- Colab bootstrap: installs deps + the neuro101 package ONLY on Colab ---
import sys, os
if "google.colab" in sys.modules:
    !pip install -q "mne==1.8.0" "moabb==1.2.0" "braindecode==0.8.1" "pyriemann==0.7" "scikit-learn==1.5.2"
    if not os.path.exists("neural-signals-101"):
        !git clone -q https://github.com/ChiShengChen/neural-signals-101
    sys.path.insert(0, os.path.abspath("neural-signals-101/src"))
    print("Colab setup complete — continue to the chapter below.")

# Chapter 01 — What Neural Signals Are

Before we filter or classify anything, let's build intuition for *what we are
even measuring* and *where the noise comes from*.

## Learning objectives
1. Name the main neural signal types and their **physical origin**.
2. Compare them on **spatial scale**, **temporal scale**, and **signal-to-noise
   ratio (SNR)** — how strong the signal is relative to the noise.
3. Look at a real EEG trace and *point at* the noise.
4. Explain, in plain words, **where the EEG signal physically comes from** —
   which cells generate it and why it is so small by the time it reaches the scalp.

> **Prerequisites:** Chapter 00.
> **Difficulty:** ★★☆☆☆

**Runtime:** ~1 min (uses one small PhysioNet subject already cached from Ch00).

In [ ]:
import sys
from pathlib import Path
try:
    import neuro101  # noqa: F401
except ModuleNotFoundError:
    sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np
import matplotlib.pyplot as plt
from neuro101 import io, datasets as ds, features as ft

## A field guide to neural signals

All of these are ways of measuring the electrical (or metabolic) activity of
neurons, but at very different scales and with very different trade-offs.

| Signal | What it measures | Where it's recorded | Spatial scale | Temporal scale | SNR |
|---|---|---|---|---|---|
| **EEG** | summed electrical activity of many neurons | scalp (non-invasive) | cm (blurry) | ms (fast) | **low** |
| **MEG** | magnetic fields from the same currents | outside the head | cm | ms | low–medium |
| **ECoG** | electrical activity from the cortical surface | on the brain (surgery) | mm | ms | high |
| **LFP** | local field potential of a small population | electrode in tissue | sub-mm | ms | high |
| **Spikes** | individual neuron action potentials | electrode next to a cell | single neuron | sub-ms | high |
| **fNIRS** | blood oxygenation (proxy for activity) | scalp (light) | cm | **seconds** (slow) | medium |
| **EMG** | muscle electrical activity | skin over muscle | muscle | ms | high |

**The core trade-off:** the less invasive the method, the more the signal is
blurred and buried in noise. EEG is easy to record but has *low SNR*; spikes
have beautiful SNR but require putting an electrode next to a neuron.

## Where EEG physically comes from

### The cells that generate the signal

The brain has billions of neurons, but EEG is almost entirely generated by one
specific type: **cortical pyramidal neurons**.  These cells are long and thin,
arranged in neat parallel columns perpendicular to the skull — think of them like
a bunch of pencils all pointing the same direction.

When a pyramidal neuron receives input from other neurons, it generates a
**post-synaptic potential (PSP)** — a slow, sustained shift in voltage that lasts
tens of milliseconds.  PSPs from thousands of neighbouring neurons, all
synchronised, **add up** into a detectable electrical field that spreads outward
through tissue.  *That* is what EEG records.

> **Why not action potentials (spikes)?**  Spikes are fast (~1 ms) and have
> opposite polarities in different parts of the same neuron, so they largely
> cancel each other out when you sum across millions of cells.  PSPs are slow
> and spatially coherent, so they survive the summing.  EEG is a "slow wave"
> signal, not a spike detector.

### Volume conduction — why nearby channels are correlated

Once that summed electrical field exists inside the brain, it doesn't stay
in one spot.  It spreads outward through brain tissue, cerebrospinal fluid,
skull, and scalp — a process called **volume conduction**.

Think of dropping a stone into a pond.  The ripple spreads in every direction.
The skull and scalp act like a blurry lens: each brain source "lights up" many
electrodes at once, and nearby electrodes pick up heavily overlapping versions of
the same sources.  This is the fundamental reason **neighbouring EEG channels are
highly correlated** — they are not independent sensors of different things.

*Forward-reference:* this spatial blurring is exactly why we cannot freely shuffle
or permute channels/samples in EEG data — doing so destroys the spatial structure
that encodes which brain region is active.  We return to this in Chapter 12.

### Why the scale is microvolts (µV)

The signal starts out as millivolt-level voltage differences inside the cortex.
By the time it crosses ~7 cm of resistive tissue (brain, CSF, skull, scalp) it
has been attenuated by roughly **a factor of 1000**, leaving us with
**tens of microvolts** at the scalp.  A microvolt is one-millionth of a volt —
smaller than the electrical noise picked up by a typical headphone cable.  This
is why EEG amplifiers must be extremely sensitive, and why even a small eye blink
(~100 µV muscle artefact) can swamp the brain signal.

### The 10-20 electrode system

To make EEG results reproducible and comparable across labs, the research
community settled on a standard scheme for placing and naming electrodes called
the **10-20 system** (the numbers refer to the percentage of head distances used
to space the electrodes).

**Reading an electrode name:**
- The **letter** tells you the brain region underneath:
  - **F** = Frontal (planning, attention)
  - **C** = Central (motor / somatosensory strip)
  - **T** = Temporal (auditory, language)
  - **P** = Parietal (spatial awareness)
  - **O** = Occipital (visual)
  - **Fp** = Frontopolar; **z** suffix = midline (zero = centre)
- The **number** tells you left vs right: odd = left hemisphere, even = right.
  - So **C3** = Central, left hemisphere; **C4** = Central, right hemisphere.
  - **Cz** = Central, midline.

Common landmarks you will see throughout this course: **Fz** (midline frontal),
**Cz** (vertex, top of the head), **C3/C4** (motor cortex, used heavily in motor
imagery BCIs), **Oz** (occipital midline, visual evoked potentials).

The visualisation below shows the standard electrode positions:

In [ ]:
try:
    import mne
    mont = mne.channels.make_standard_montage('standard_1020')
    fig_mont = mont.plot(show=False)
    fig_mont.suptitle("Standard 10-20 electrode positions\n"
                      "(letters = brain region, odd = left, even = right)",
                      fontsize=10)
    plt.show()
except Exception:
    # Fallback: draw a simple schematic with matplotlib
    fig, ax = plt.subplots(figsize=(5, 5))
    circle = plt.Circle((0, 0), 1, color="#e8d5b7", ec="#333", lw=2)
    ax.add_patch(circle)
    # Nose indicator
    ax.annotate("", xy=(0, 1.15), xytext=(0, 1.0),
                arrowprops=dict(arrowstyle="-", color="#333", lw=2))
    electrodes = {
        "Fz": (0.0, 0.5), "Cz": (0.0, 0.0), "Oz": (0.0, -0.6),
        "C3": (-0.5, 0.0), "C4": (0.5, 0.0),
        "F3": (-0.35, 0.45), "F4": (0.35, 0.45),
        "T7": (-0.85, 0.0), "T8": (0.85, 0.0),
    }
    for name, (x, y) in electrodes.items():
        ax.plot(x, y, "o", ms=12, color="#4c72b0", zorder=3)
        ax.text(x, y + 0.12, name, ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.4)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title("Schematic: 10-20 electrode positions\n"
                 "(letters = brain region, odd = left, even = right)",
                 fontsize=10)
    plt.show()

## Orders of magnitude (why units matter)

- **EEG**: tens of **microvolts** (µV, millionths of a volt). A blink is ~100 µV
  and *dwarfs* the brain signal you care about.
- **Spikes**: ~100 µV but at the electrode tip, sharp and brief (~1 ms).
- **fNIRS**: changes over **seconds**, because blood flow is slow.

Mixing these scales up (e.g. treating EEG like an audio signal) is a classic
beginner error. Let's look at real EEG.

In [ ]:
X, y, subj = io.load_physionet_mi_epochs(n_subjects=1)
sf = ds.DATASETS["physionet_mi"].sfreq_hz
print(f"Loaded {X.shape[0]} epochs, {X.shape[1]} channels, {X.shape[2]} samples each @ {sf} Hz")

## Where is the noise? Look at one channel

Real EEG is a sum of: (1) brain rhythms you want, (2) **biological artefacts**
(eye blinks, muscle, heartbeat), and (3) **environmental noise** (50/60 Hz mains
hum, electrode drift). The plot below shows one channel; the annotations point
at the usual suspects.

In [ ]:
trial = X[0, 0] * 1e6  # convert to µV for readable axis
t = np.arange(trial.size) / sf

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t, trial, lw=0.8, color="#333")
ax.set(xlabel="Time (s)", ylabel="Amplitude (µV)",
       title="One EEG channel — a mix of brain signal and noise")
ax.axhline(0, color="gray", lw=0.5)
plt.show()

## Make the noise visible: where does the power live?

A quick way to "see" noise is to look at how power is spread across frequencies.
Brain rhythms cluster at low frequencies (< ~30 Hz); muscle artefacts spread to
high frequencies; mains hum is a sharp spike at exactly 50 or 60 Hz.

### ✏️ Predict before you run

Before executing the cell below, take 30 seconds and write down your predictions:

1. **Where will the power be concentrated?**  Low frequencies (< 30 Hz), mid
   frequencies (30–70 Hz), high frequencies (> 70 Hz), or spread evenly?
2. **Will you see a sharp spike at 50 Hz or 60 Hz?**  (Hint: PhysioNet is a US
   dataset.  Which mains frequency does the US use?)
3. **Will the spectrum look flat ("white") or fall off at high frequencies?**

Once you have a guess, run the cell and compare.  Actively predicting and then
checking is one of the fastest ways to build intuition.

In [ ]:
from scipy.signal import welch
freqs, psd = welch(X[:, 0, :], fs=sf, nperseg=int(sf))  # PSD per epoch, channel 0
mean_psd = psd.mean(axis=0)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.semilogy(freqs, mean_psd, color="#c44e52")
for f0, name in [(10, "alpha (~10 Hz)\nbrain rhythm"), (60, "60 Hz\nmains hum (US)")]:
    ax.axvline(f0, ls="--", color="gray", lw=1)
    ax.text(f0 + 1, mean_psd.max() * 0.3, name, fontsize=8)
ax.set(xlim=(0, 80), xlabel="Frequency (Hz)", ylabel="Power (log)",
       title="Power spectrum — brain rhythms vs noise")
plt.show()

The hump around ~10 Hz is the **alpha rhythm** (brain). Any sharp line at 50/60
Hz is **mains noise** (we remove it with a notch filter in Chapter 04). Broadband
power climbing at high frequency is often **muscle (EMG)** contamination.

## ✅ Concept check

Try to answer these without scrolling back up.  Then check your answers below.

**Q1.** Why are neighbouring EEG channels highly correlated with each other?

**Q2.** EEG amplitude is measured in microvolts (µV), not millivolts (mV).
Why is it so small by the time it reaches the scalp electrodes?

**Q3.** What does the electrode label **C3** mean in the 10-20 system?
Which hemisphere is it over, and what brain region does the letter "C" refer to?

---

**Answers:**

**A1.** Because of **volume conduction**: every brain source spreads its
electrical field outward through skull and scalp, so a single source
activates many electrodes at once.  Nearby electrodes pick up heavily
overlapping mixtures of the same underlying sources — they are *not*
independent sensors of completely different things.

**A2.** The signal is attenuated by roughly **1000×** as it passes through
~7 cm of resistive tissue (brain, cerebrospinal fluid, skull, scalp).
A millivolt-scale cortical signal arrives at the electrode as only tens
of microvolts — smaller than the noise on a typical audio cable.

**A3.** **C** = Central (the motor / somatosensory strip region);
**3** = odd number = *left* hemisphere.  So C3 sits over the left motor
cortex — the region that controls the *right* hand.  (C4 is the mirror:
right hemisphere, controls the left hand.  Cz is the midline vertex.)

## ⚠️ Common mistakes / why this is wrong

- **Assuming a clean signal.** EEG is *mostly* noise from your model's point of
  view. The single biggest determinant of BCI accuracy is often artefact
  handling, not the classifier.
- **Confusing temporal scales.** fNIRS changes over seconds; spikes over a
  millisecond. A method (or sampling rate) tuned for one is wrong for another.
- **Treating channels as independent sensors of different things.** Nearby EEG
  electrodes see *overlapping* sources (volume conduction) — they are highly
  correlated. This is exactly why we cannot shuffle samples freely later.
- **Ignoring units.** Reporting "amplitude 0.00007" instead of "70 µV" hides
  bugs. Keep physical units; sanity-check that EEG is tens of µV.

**Next:** Chapter 02 — *ML from zero*: before we touch signals with models, we
build the machine-learning reflexes (overfitting, why the test set is sacred)
that the rest of the tutorial depends on.